# ANGSI JR. V9.3.1 — Launch in JupyterLab
Run these cells on the HPC where Angsi seismic files are stored. No `%run app.py` or `python app.py` inside a notebook cell.


In [ ]:
from pathlib import Path
import os, sys, socket, py_compile
ROOT=Path('/data/epic20/APGCE/apgce_team_13')
APP=ROOT/'ANGSI_JR_V9_3_1_FACIES'/'angsi_jr_v9.py'
assert APP.is_file(), f'App missing: {APP}'
py_compile.compile(str(APP),doraise=True)
print('Python syntax PASS | Host',socket.gethostname(),'| App',APP)


## Install dependencies in the notebook interpreter


In [ ]:
%pip install -r /data/epic20/APGCE/apgce_team_13/ANGSI_JR_V9_3_1_FACIES/requirements.txt


## Start server on port 8516 (V9.3.1)


In [ ]:
import subprocess, sys, os
from pathlib import Path
log=APP.parent/'server_v931.log'
env=os.environ.copy()
env['ANGSI_DATA_ROOT']='/data/epic20/APGCE/input_data/challenge3'
with log.open('a') as handle:
    process_v931=subprocess.Popen(
        [sys.executable,'-m','streamlit','run',str(APP),
         '--server.address=127.0.0.1','--server.port=8516',
         '--server.headless=true','--server.fileWatcherType=none'],
        cwd=str(APP.parent),env=env,stdout=handle,
        stderr=subprocess.STDOUT,start_new_session=True)
print('PID',process_v931.pid,'LOG',log)


## Server health check and diagnostics


In [ ]:
import urllib.request
try:
    with urllib.request.urlopen('http://127.0.0.1:8516/_stcore/health',timeout=5) as resp:
        print('HTTP',resp.status, resp.read().decode())
except Exception as e:
    print('Not healthy:',repr(e))
print('Recent log:')
print(log.read_text(errors='replace')[-5000:] if log.exists() else 'No log yet')


## Open dashboard
In Firefox on the same HPC remote desktop: `http://127.0.0.1:8516`. From an outside browser use an available JupyterHub proxy or IT-approved forwarding; `localhost` is always relative to the browser machine.
